# 데이터 전처리 (Data Preprocessing)

> 분석이나 머신러닝 사용하기 전 데이터를 깨끗하게 정리하는 모든작업

## Why?

현실 세계에서 수집한 데이터는 거의 항상 더럽다 
ex) 예를 들면 시스템 오류로 같은 데이터가 두번 이상 처리되거나 휴먼 에러가 생성하기  때문


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#더러운 데이터

raw_df=pd.DataFrame({
    'name':['Kim','Lee','Park','Choi','Choi','Jung','Han','Oh'],
    'study_hours':[2,4,None,6,6,7,100,5],
    'attendance':[60,75,80,None,None,90,95,70],
    'score':[62,78,82,85,85,None,99,74]
})

raw_df

In [ ]:
# 셀 단위로 NaN 인지 여부를 True/False로 표시
raw_df.isna()

In [ ]:
raw_df.isna().sum()

## 결측값 처리 : 평균값으로 채우기

> 평균값 채우기는 정답이 아니라 시작점을 의미한다 0, 중앙값(50%), 직전값, 모델 예측값을 사용한다 

In [ ]:
# 원본

filled_df=raw_df.copy()

filled_df['study_hours']=filled_df['study_hours'].fillna(filled_df['study_hours'].mean())
filled_df['attendance']=filled_df['attendance'].fillna(filled_df['attendance'].mean())
filled_df['score']=filled_df['score'].fillna(filled_df['score'].mean())

filled_df

## 결측값 처리: 결측값이 있는 행 제거하기

**when**
- 결측 비율이 낮을때 (ex 전체 1~5% 이하)
- 그 행을 잃어도 분석 결과에 큰 영향이 없을때
- 조금이라도 비어 있는 데이터는 신뢰하지 않을때 (보수적 정책)



In [ ]:
# dropna 는 결측값이 하나라도 있는 행을 통째로 삭제하는 함수

dropped_df=raw_df.dropna()
dropped_df

## 중복 데이터 제거하기

같은 행이 두번 들어가있으면 한사람이 두명처럼 카운트되어 평균/합계/비율 같은 집계가 모두 왜곡이 된다

In [ ]:
filled_df.duplicated()

In [ ]:
no_duplicated_df=filled_df.drop_duplicates()
no_duplicated_df

## 문자열 공백 제거하기

"Kim"==" Kim" -> False

In [ ]:
clean_name_df=no_duplicated_df.copy()
clean_name_df["name"]=clean_name_df['name'].str.strip()
clean_name_df

# series.str.strip() - 앞뒤 공백제거
# series.str.lstrip() - 앞 공백제거
# series.str.rstrip() - 뒤 공백제거
# series.str.replace() - 모든 공백제거


## 이상치(outlier) 확인하기

> 다른 값들과 비교했을때 지나치게 크거나 작은 값
1. 입력오류 측정오류 - 단위를 잘못 적었거나 센서가 튄 경우 -> 제거/수정 하는것이 옳다
2. 특이한 사례 - 압도적으로 점수가 높거나/낮거나 고액 결제

In [ ]:
clean_name_df.describe()

In [ ]:
plt.Figure(figsize=(5,4))
plt.boxplot(clean_name_df['study_hours'])
plt.title('Study Hours Boxplot')
plt.ylabel('Study Hours')
plt.show()

# IQR로 이상치 제거하기
IQR(Interquratille Range): 데이터를 정렬했을때 가운데 50%가 차지하는 범위

**계산절차**
1. Q1 = 하위 25% 지점의 값
2. Q2 = 상위 25% 지점의 값
3. IQR = Q3 - Q1 (가운데 50%의 폭)
4. 정상 범위 = [Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]
5. 이 범위를 벗어나면 이상치로 간주

**왜 1.5를 사용하는가**
정규분포에서 위 범위 바깥에 떨어질 확률이 75% 약 0.7%로 매우 낮기 때문에 통계 학자가 BoxPlot을 만들면서 제안한 경험적 기준

- 1.5 -> 일반적으로 쓰는 기준
- 3.0 -> 보수적인 기준 (극단적인 값만 골라냄)


In [ ]:
q1=clean_name_df['study_hours'].quantile(0.25)
q3=clean_name_df['study_hours'].quantile(0.75)
iqr=q3-q1

lower=q1-1.5*iqr
upper=q3+1.5*iqr

print('q1',q1)
print('q3',q3)
print("IQR",iqr)
print("정상범위",lower,'~',upper)

In [ ]:
clean_df=clean_name_df[
    (clean_name_df['study_hours']>= lower) &
    (clean_name_df['study_hours']<= upper)
]

clean_df

## 전처리 후 새열 만들기
- 데이터를 깨끗이 정리한 뒤에야 의미 있는 분석이 가능하다.
- 점수(score)가 75점 기준으로 합격/불합격을 표시하는 pass_fail 열을 추가하기

In [ ]:
final_df=clean_df.copy()
final_df['pass_fail']=final_df['score'].apply(lambda x: 1 if x>= 75 else 0)
final_df


**문제**

실습용 매출데이터
- product:앞뒤 공백 섞임
- price:결측 값이 2개
- quantity:결측값이 1개

sales_raw=pd.DataFrame({
    'product':["Apple","Banana","Orange","Orange","Milk","Bread",]
    'price':[1200,800,None,None,1650,2000]
    'quantity':[5,3,4,4,None,2]
})

In [ ]:
sales_raw=pd.DataFrame({
    'product':[" Apple","Banana ","Orange","Orange","Milk","Bread",],
    'price':[1200,800,None,None,1650,2000],
    'quantity':[5,3,4,4,None,2]
})

print("=== 원본 데이터 ===")
print(sales_raw)

print("\n=== 결측값 확인 (isna) ===")
print(sales_raw.isna())

print("\n=== 결측값 개수 (isna().sum()) ===")
print(sales_raw.isna().sum())

sales_df = sales_raw.copy()

sales_df['product'] = sales_df['product'].str.strip()

sales_df['price'] = sales_df['price'].fillna(sales_df['price'].mean())

sales_df['quantity'] = sales_df['quantity'].fillna(sales_df['quantity'].mean())

print("\n=== 결측값 처리 후 ===")
print(sales_df)

print("\n=== 중복 확인 (duplicated) ===")
print(sales_df.duplicated())

sales_df = sales_df.drop_duplicates()

print("\n=== 최종 전처리 결과 ===")
print(sales_df)